# Chat with Angel and Devil

In [ ]:
import base64
import openai
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from io import BytesIO
from PIL import Image

load_dotenv(override=True)


PLACEHOLDER = "PLACEHOLDER"
MODELS = [ "gpt-5-nano", "gpt-3.5-turbo", "o4-mini"]

In [ ]:
user_system_prompt = """
You're a user who seeks advice from both Angel and Devil. What's on your mind today?
Always respond in the following format (without ```) and use plain text for your message only:
```
### USER
{message}
```
"""

user_user_prompt = f"""
You are in a conversation with Angel and Devil discussing your problems.
The conversation so far is as follows:
{PLACEHOLDER}
Now with this, respond with what you would like to say next, as the user.
"""

angel_system_prompt = """
You are Angel, who comforts everyone and finds words of encouragement and hope.
You always try to make the best of things and be kind.
When someone asks you for advice, you always suggest what you yourself would do in that situation as Angel.
Always respond in the following format (without ```) and use plain text for your message only:
```
### ANGEL
{message}
```
"""

angel_user_prompt = f"""
You are Angel in a conversation with a user while there is also an intruding Devil talking to the user.
The conversation so far is as follows:
{PLACEHOLDER}
Now with this, respond with what you would like to say next, as Angel.
"""

devil_system_prompt = """
You are Devil, who is always cynical and takes pleasure in tormenting others.
When someone asks you for advice, you always suggest what you yourself would do in that situation as Devil.
Always respond in the following format (without ```) and use plain text for your message only:
```
### DEVIL
{message}
```
"""

devil_user_prompt = f"""
You are Devil in a conversation with a user while there is also an intruding Angel talking to the user.
The conversation so far is as follows:
{PLACEHOLDER}
Now with this, respond with what you would like to say next, as Devil.
"""

In [ ]:
def call(system_prompt, user_prompt, history, model=MODELS[0], stream=True):
    print(f"Calling {model} {"with" if stream else "without"} streaming")
    messages = [{"role": "system", "content": system_prompt}]
    if len(history) > 0:
        messages.append({"role": "user", "content": user_prompt.replace(PLACEHOLDER, str(history))})

    return openai.chat.completions.create(model=model, messages=messages, stream=stream)

def map_no_stream(response):
    return response.choices[0].message.content or ""

def display_no_stream(response):
    message = map_no_stream(response)
    display(Markdown(message))
    return message

def map_stream(chunk):
    return chunk.choices[0].delta.content or ""

def display_stream(stream):
    result = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        result += map_stream(chunk)
        update_display(Markdown(result), display_id=display_handle.display_id)
    return result

def display_result(content, stream):
    if stream:
        return display_stream(content)
    else:
        return display_no_stream(content)

In [ ]:
def image(prompt):
    image_response = openai.images.generate(
            model="gpt-image-1-mini",
            prompt=f"An image representing a the persona described with this system prompt: {prompt} without any text",
            size="1024x1024",
            n=1,
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

# CLI-Version

In [ ]:
def chat(stream, participate, max_len = 5):
    history = []
    for i in range(max_len):
        if participate:
            message = f"### USER\n{input()}\n"
            display(Markdown(message))
        else:
            message_response = call(user_system_prompt, user_user_prompt, history, stream)
            message = display_result(message_response, stream)
        history.append({ "role": "user", "content": message})

        angel_response = call(angel_system_prompt, angel_user_prompt, history, stream)
        angel_answer = display_result(angel_response, stream)
        history.append({ "role": "angel", "content": angel_answer})

        devil_response = call(devil_system_prompt, devil_user_prompt, history, stream)
        devil_answer = display_result(devil_response, stream)
        history.append({ "role": "angel", "content": devil_answer})


In [ ]:
chat(stream=True, participate=False) # set participate=True to be part of the conversation

## Gradio

In [ ]:
import gradio as gr

In [ ]:
def gr_handle_message(response, stream):
    if stream:
        result = ""
        for chunk in response:
            result += map_stream(chunk)
            yield result
    else:
        yield map_no_stream(response)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

def chat_gradio(history, model, stream):
    history = [{ "role": h["role"], "content": h["content"]} for h in history]
    history.append({ "role": "assistant", "content": ""})
    
    angel_response = call(angel_system_prompt, angel_user_prompt, history, model, stream)
    for chunk in gr_handle_message(angel_response, stream):
        history[-1]["content"] = chunk
        yield history[-2:]

    history.append({ "role": "assistant", "content": ""})

    devil_response = call(devil_system_prompt, devil_user_prompt, history, model, stream)
    for chunk in gr_handle_message(devil_response, stream):
        history[-1]["content"] = chunk
        yield history[-3:]

In [ ]:
angel_image = image(angel_system_prompt)
display(angel_image)

In [ ]:
devil_image = image(devil_system_prompt)
display(devil_image)

In [ ]:
with gr.Blocks(title="Chat with Angel and Devil", fill_width=True, fill_height=True) as ui:
    with gr.Row():
        gr.Markdown("# Chat with Angel and Devil")
    with gr.Row():
        with gr.Column(scale=1):
            gr.Image(angel_image, label="Angel", container=False, show_fullscreen_button=False, show_download_button=False)
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(elem_id="chatbot", type="messages", group_consecutive_messages=False, show_label=False)
        with gr.Column(scale=1):
            gr.Image(devil_image, label="Devil", container=False, show_fullscreen_button=False, show_download_button=False)
    with gr.Row():
        message = gr.Textbox(show_label=False, placeholder="Type a message...")
    with gr.Row():
        with gr.Accordion(label="Settings"):
            stream_selector = gr.Checkbox(value=True, label="Stream Responses")
            model_selector = gr.Dropdown(MODELS, label="Select model", value=MODELS[0])

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat_gradio, inputs=[chatbot, model_selector, stream_selector], outputs=chatbot
    )

ui.launch()